# ISIC-2024 baseline reproduction (legacy-v2)

This notebook reproduces the manuscript performance protocol only: 40 epochs, fixed patient split (`split_seed=42`), train-only 1:20 subsampling, final-epoch checkpoint, and three model seeds. It does **not** run OOD exposure or make TensorRT-speed claims. Enable Kaggle Internet for the GitHub pull; the setup first uses attached inputs and otherwise asks KaggleHub to attach/download them.

In [ ]:
# Cell 1 -- clone/pull code and resolve the real ISIC input
import os, sys, json, subprocess, shlex, time
from pathlib import Path

REPO_URL = 'https://github.com/minhduc110207/MDEP-Microglial-Driven-Evidential-Pruning.git'
REPO_BRANCH = 'main'  # replace with an immutable commit/branch if required by the paper artifact
REPO_ROOT = Path('/kaggle/working/MDEP-Microglial-Driven-Evidential-Pruning')
WORK = Path('/kaggle/working')
OUT = WORK / 'paper_experiment_outputs'
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print('\n$', shlex.join(cmd), flush=True)
    merged = os.environ.copy(); merged.update({str(k): str(v) for k, v in (env or {}).items()})
    subprocess.run(cmd, cwd=str(cwd or REPO_ROOT), env=merged, check=True)

if not (REPO_ROOT / '.git').exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_ROOT], cwd=WORK)
else:
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_ROOT)

def find_isic(root):
    root = Path(root)
    for csv in [root / 'train-metadata.csv', *root.rglob('train-metadata.csv')]:
        parent = csv.parent
        if (parent / 'train-image.hdf5').exists() or (parent / 'train-image' / 'image').exists():
            return parent
    return None

isic_candidates = [Path('/kaggle/input/competitions/isic-2024-challenge'), Path('/kaggle/input/isic-2024-challenge')]
ISIC_ROOT = next((p for p in (find_isic(x) for x in isic_candidates) if p), None)
if ISIC_ROOT is None:
    try:
        import kagglehub
        ISIC_ROOT = find_isic(kagglehub.competition_download('isic-2024-challenge'))
    except Exception as exc:
        raise RuntimeError('ISIC-2024 was not available. Accept the competition rules, enable Internet, or attach the competition input via Add Data.') from exc
if ISIC_ROOT is None:
    raise RuntimeError('Downloaded/attached ISIC input has no train-metadata.csv plus train images.')
os.environ['ISIC_ROOT'] = str(ISIC_ROOT)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
assert (REPO_ROOT / 'experiments/isic_paper_experiments.py').exists()
print('[OK] commit =', commit)
print('[OK] ISIC_ROOT =', ISIC_ROOT)


In [ ]:
# Cell 2 -- all published baselines plus the Full GUDS reference; serial and resume-safe
SEEDS = (42, 123, 456)
SPLIT_SEED, EPOCHS, BATCH_SIZE, LR = 42, 40, 32, 4e-5
PROFILE, SUFFIX = 'legacy_v2', '_paper_v2_repro'
MODELS = (
    'full_guds', 'standard_ce', 'focal_loss', 'class_balanced_ce', 'balanced_softmax',
    'ldam_drw', 'decoupled_crt', 'mislas', 'dense_edl', 'fisher_edl',
    'flexible_edl', 'r_edl', 'static_24_edl', 'rigl_style_24',
)

def metrics_path(name, seed):
    return OUT / 'isic' / f'{name}{SUFFIX}' / f'seed_{seed}' / 'metrics.json'

def completed(name, seed):
    path = metrics_path(name, seed)
    try:
        d = json.loads(path.read_text())
        return (d.get('protocol_profile') == PROFILE and d.get('epochs') == EPOCHS and
                d.get('split_seed') == SPLIT_SEED and d.get('experiment', {}).get('base_name') == name and
                d.get('model_seed', d.get('seed')) == seed and d.get('checkpoint_selection', {}).get('metric') == 'last')
    except (OSError, ValueError, TypeError):
        return False

for name in MODELS:
    for seed in SEEDS:
        if completed(name, seed):
            print(f'[SKIP] {name}, seed={seed}: validated result already exists')
            continue
        cmd = [sys.executable, '-u', 'experiments/isic_paper_experiments.py',
               '--experiment', name, '--seed', str(seed), '--split_seed', str(SPLIT_SEED),
               '--epochs', str(EPOCHS), '--batch_size', str(BATCH_SIZE), '--lr', str(LR),
               '--subsample_scope', 'train', '--subsample_ratio', '20',
               '--checkpoint_selection', 'last', '--structural_proxy_batches', '4',
               '--efl_gamma_final', '0.0', '--protocol_profile', PROFILE, '--run_suffix', SUFFIX]
        # Each model uses a fresh subprocess: no GPU/RAM accumulation between baseline runs.
        run(cmd, cwd=REPO_ROOT, env={'ISIC_ROOT': ISIC_ROOT, 'MDEP_DETERMINISTIC': '1'})

missing = [(n, s) for n in MODELS for s in SEEDS if not completed(n, s)]
assert not missing, f'Incomplete or protocol-mismatched runs: {missing}'
print('[DONE] All baseline and reference runs passed the protocol gate.')


In [ ]:
# Cell 3 -- export a compact, seed-level audit file for download as Kaggle output
rows = []
for name in MODELS:
    for seed in SEEDS:
        d = json.loads(metrics_path(name, seed).read_text())
        m = d.get('metrics', {})
        rows.append({'experiment': name, 'seed': seed, 'pauc': m.get('pauc'),
                     'macro_auroc': m.get('macro_auroc'), 'pr_auc': m.get('pr_auc'),
                     'protocol_profile': d.get('protocol_profile'),
                     'checkpoint_selection': d.get('checkpoint_selection', {}).get('metric')})
manifest = {'git_commit': commit, 'isic_root': str(ISIC_ROOT), 'seeds': SEEDS, 'split_seed': SPLIT_SEED,
            'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR, 'subsample_scope': 'train',
            'subsample_ratio': 20, 'checkpoint_selection': 'last', 'protocol_profile': PROFILE,
            'models': MODELS, 'note': 'Legacy-v2 performance reproduction; not a TensorRT sparse-kernel benchmark.'}
(OUT / 'baseline_legacy_v2_manifest.json').write_text(json.dumps(manifest, indent=2))
import pandas as pd
table = pd.DataFrame(rows)
table.to_csv(OUT / 'baseline_legacy_v2_seed_metrics.csv', index=False)
display(table.groupby('experiment')[['pauc', 'macro_auroc', 'pr_auc']].agg(['mean', 'std']).sort_values(('pauc', 'mean'), ascending=False))
print('[SAVED]', OUT / 'baseline_legacy_v2_seed_metrics.csv')
